# ChemBreak V8
## Standard Google Colab + Vertex AI

This notebook runs the independent `ChemBreak_V8_Cloud` codebase from ordinary Google Colab.

It does not use V7 or any earlier ChemBreak version.

The notebook itself is only the controller. Gemini, Llama 4 Maverick, and gpt-oss-120B are still called through your Vertex AI project, so you do not need an A100 runtime for this version.


## 1. Authenticate to Google Cloud

Sign in with the same Google account that has access to the project `rs-foundsecft-mghasemi`.

No personal Vertex API key is required.


In [ ]:
from google.colab import auth
auth.authenticate_user()

from pathlib import Path
import subprocess, sys, json, os, shutil

PROJECT_ID = "rs-foundsecft-mghasemi"

subprocess.run(
    ["gcloud", "config", "set", "project", PROJECT_ID],
    check=True
)

print("Authenticated to Google Cloud project:", PROJECT_ID)


## 2. Mount Google Drive for resumable output

Recommended. Test, pilot, and production outputs will be stored separately in Drive.


In [ ]:
USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    STORAGE_ROOT = Path("/content/drive/MyDrive/ChemBreak_V8")
else:
    STORAGE_ROOT = Path("/content/ChemBreak_V8")

STORAGE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent storage root:", STORAGE_ROOT)


## 3. Clone or refresh your GitHub repository

This notebook uses only the `ChemBreak_V8_Cloud` folder in the repository.

Deleting V7 later will not affect this notebook.


In [ ]:
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
REPO_ROOT = Path("/content/ChemBreak_repo")
PROJECT_SUBDIR = "ChemBreak_V8_Cloud"

if (REPO_ROOT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
        check=True
    )
elif REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

PROJECT_DIR = REPO_ROOT / PROJECT_SUBDIR

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"{PROJECT_SUBDIR} was not found in the GitHub repository. "
        "Upload the full V8 Cloud folder to GitHub first."
    )

PIPELINE = PROJECT_DIR / "scripts" / "chembreak_v8_cloud.py"
CONFIG_SOURCE = PROJECT_DIR / "config" / "run_config.json"

print("V8 folder:", PROJECT_DIR)
print("Pipeline:", PIPELINE)


## 4. Install the V8 requirements


In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(PROJECT_DIR / "requirements.txt")
    ],
    check=True
)
print("Requirements installed.")


## 5. Choose the run

Start with `test`.

- test: 9 final tasks
- pilot: 100 final tasks plus reserve assignments
- production: 500 final tasks plus reserve assignments


In [ ]:
RUN_TYPE = "test"   # "test", "pilot", or "production"

# Optional. Leave blank when using Drive persistence.
GCS_OUTPUT_URI = ""

RUNTIME_DIR = Path("/content/ChemBreak_V8_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG = RUNTIME_DIR / f"run_config_{RUN_TYPE}.json"

cfg = json.loads(CONFIG_SOURCE.read_text(encoding="utf-8"))
cfg["run_type"] = RUN_TYPE
cfg["project_id"] = PROJECT_ID
cfg["gcs_output_uri"] = GCS_OUTPUT_URI

RUNTIME_CONFIG.write_text(
    json.dumps(cfg, indent=2),
    encoding="utf-8"
)

OUTPUT_DIR = STORAGE_ROOT / "outputs" / RUN_TYPE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_stage(stage):
    command = [
        sys.executable, "-u", str(PIPELINE),
        "--stage", stage,
        "--project-dir", str(PROJECT_DIR),
        "--config", str(RUNTIME_CONFIG),
        "--output-dir", str(OUTPUT_DIR),
    ]
    print(f"\n===== {stage.upper()} =====\n", flush=True)
    subprocess.run(command, check=True)
    print(f"\nCompleted: {stage}", flush=True)

print("Run type:", RUN_TYPE)
print("Output directory:", OUTPUT_DIR)


## 6. Preflight model access

This sends harmless JSON requests only. Do not continue to generation unless the required models report `OK`.


In [ ]:
run_stage("preflight")

import pandas as pd
from IPython.display import display

display(pd.read_csv(OUTPUT_DIR / "preflight_models.csv"))


## 7. Bootstrap fresh source provenance

No V7 task bank is used.


In [ ]:
run_stage("bootstrap")


## 8. Build the fresh V8 assignment plan


In [ ]:
run_stage("plan")


## 9. Inspect coverage before generation


In [ ]:
plan = pd.read_csv(OUTPUT_DIR / "assignments_v8.csv")
display(plan.head(20))

print("Assignments:", len(plan))

print("\nHC coverage")
display(plan["hc_id"].value_counts().sort_index())

print("\nHD coverage")
display(plan["hd_id"].value_counts().sort_index())

print("\nOT coverage")
display(plan["ot_id"].value_counts().sort_index())


## 10. Generate the candidate pool

Default generators:
- Gemini 3.1 Pro Preview
- Llama 4 Maverick
- gpt-oss-120B


In [ ]:
run_stage("generate")


## 11. Deterministic validation


In [ ]:
run_stage("validate")


## 12. Repair invalid candidates


In [ ]:
run_stage("repair")


## 13. Blind judging

Judge 1: Gemini 2.5 Pro  
Judge 2: gpt-oss-120B  
Adjudicator: Gemini 3.1 Pro Preview


In [ ]:
run_stage("judge")


## 14. Refill unresolved assignments and judge again


In [ ]:
run_stage("refill")
run_stage("judge")


## 15. Finalize and inspect outputs


In [ ]:
run_stage("finalize")
run_stage("status")

for name in [
    "run_summary.json",
    "coverage_report.csv",
    "diversity_report.csv",
    "final_task_bank.csv",
]:
    path = OUTPUT_DIR / name
    print("\n", name)
    if path.suffix == ".json" and path.exists():
        print(path.read_text(encoding="utf-8"))
    elif path.exists():
        display(pd.read_csv(path).head(25))
    else:
        print("not written")


## 16. Create a checkpoint ZIP in Google Drive


In [ ]:
summary = json.loads(
    (OUTPUT_DIR / "run_summary.json").read_text(encoding="utf-8")
)
label = summary["completion_label"]

archive = shutil.make_archive(
    str(STORAGE_ROOT / f"ChemBreak_V8_{RUN_TYPE}_{label}"),
    "zip",
    root_dir=str(OUTPUT_DIR)
)

print("Checkpoint ZIP:", archive)


## Recommended progression

Complete `test` first. Inspect the 9 tasks, validation results, judgments, and diversity report.

Then change `RUN_TYPE` to `pilot`, rerun from the run-selection cell, and review the 100-task pilot.

Only after pilot review should you switch to `production`.
